Load config

In [0]:
%run ../config/config

In [0]:
pipeline_table=f"{catalog}.{pipeline_schema}.pipeline_control"

In [0]:
dbutils.widgets.text("batch_id", "") 
batch_id = dbutils.widgets.get("batch_id")

Mark the current batch's "in_progress" row as "completed" via merge statement

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

if batch_id:
    delta_table = DeltaTable.forName(spark, pipeline_table)

    source_df = (
        spark.createDataFrame([(batch_id,)], ["batch_id"])
            .withColumn("status", F.lit("completed"))
            .withColumn("updated_timestamp", F.current_timestamp())
    )

    (
        delta_table.alias("t")
            .merge(
                source_df.alias("s"),
                "t.batch_id = s.batch_id AND t.status = 'in_progress'"
            )
            .whenMatchedUpdate(set={
                "status": "s.status",
                "updated_timestamp": "s.updated_timestamp"
            })
            .execute()
    )

    print(f"Batch # {batch_id} is completed")
else:
    raise Exception("batch_id is missing")  

Batch # 2022-1 is completed
